In [7]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import dcc, html, Input, Output

# =========================
# Paths
# =========================

trip_path = r"Data 2\Data 2\2024-12-11 12-00-00 (1)"
infra_path = r"Data 1"

gps_lat_file = os.path.join(trip_path, "GPS.latitude.csv")
gps_lon_file = os.path.join(trip_path, "GPS.longitude.csv")
gps_speed_file = os.path.join(trip_path, "GPS.speed.csv")
gps_sat_file = os.path.join(trip_path, "GPS.satellites.csv")

vib1_file = os.path.join(trip_path, "CH1_ACCEL1Z1.csv")
vib2_file = os.path.join(trip_path, "CH2_ACCEL1Z2.csv")

bridge_file = os.path.join(infra_path, "converted_coordinates_Resultat_Bridge.csv")
railjoint_file = os.path.join(infra_path, "converted_coordinates_Resultat_RailJoint.csv")
turnout_file = os.path.join(infra_path, "converted_coordinates_Turnout.csv")
# =========================

# =========================
# Load GPS data
# =========================

lat_df = pd.read_csv(gps_lat_file, header=None, names=["Latitude"])
lon_df = pd.read_csv(gps_lon_file, header=None, names=["Longitude"])
speed_df = pd.read_csv(gps_speed_file, header=None, names=["Speed"])
sat_df = pd.read_csv(gps_sat_file, header=None, names=["Satellites"])

df_gps = pd.concat([lat_df, lon_df, speed_df, sat_df], axis=1)
df_gps = df_gps.apply(pd.to_numeric, errors="coerce")
df_gps = df_gps.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

# GPS timing: 20 Hz => 0.05 s
df_gps["PointIndex"] = df_gps.index
df_gps["gps_time"] = df_gps.index * 0.05

print("GPS shape:", df_gps.shape)
print(df_gps.head())
# =========================

# =========================
# Load vibration data
# =========================

vib1_df = pd.read_csv(vib1_file, header=None, names=["vibration1"])
vib2_df = pd.read_csv(vib2_file, header=None, names=["vibration2"])

df_vibration = pd.concat([vib1_df, vib2_df], axis=1)
df_vibration = df_vibration.apply(pd.to_numeric, errors="coerce")
df_vibration = df_vibration.dropna().reset_index(drop=True)

# Vibration timing: 500 Hz => 0.002 s
df_vibration["timestamp"] = df_vibration.index * 0.002

print("Vibration shape:", df_vibration.shape)
print(df_vibration.head())
# =========================



# =========================
# Load infrastructure data
# =========================

def load_infra_file(file_path, label):
    df = pd.read_csv(file_path, encoding="utf-8")
    df.columns = df.columns.str.strip()
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
    df = df.dropna(subset=["Latitude", "Longitude"]).copy()
    df["Category"] = label
    return df[["Latitude", "Longitude", "Category"]]

bridge_df = load_infra_file(bridge_file, "Bridge")
railjoint_df = load_infra_file(railjoint_file, "Rail-joint")
turnout_df = load_infra_file(turnout_file, "Turnout")

infra_df = pd.concat([bridge_df, railjoint_df, turnout_df], ignore_index=True)

print("Infrastructure shape:", infra_df.shape)
print(infra_df["Category"].value_counts())
# =========================

# =========================
# Filter infrastructure points actually travelled
# =========================

def find_nearest_gps_point(lat, lon, gps_df):
    distances = np.sqrt((gps_df["Latitude"] - lat)**2 + (gps_df["Longitude"] - lon)**2)
    nearest_idx = distances.idxmin()
    nearest_distance = distances.loc[nearest_idx]
    return nearest_idx, nearest_distance

matched_rows = []

# Threshold in coordinate degrees
# Start with this. We can adjust later if needed.
distance_threshold = 0.002

for _, infra_row in infra_df.iterrows():
    nearest_idx, nearest_distance = find_nearest_gps_point(
        infra_row["Latitude"],
        infra_row["Longitude"],
        df_gps
    )

    if nearest_distance <= distance_threshold:
        matched_rows.append({
            "Latitude": infra_row["Latitude"],
            "Longitude": infra_row["Longitude"],
            "Category": infra_row["Category"],
            "NearestGPSIndex": nearest_idx,
            "NearestDistance": nearest_distance,
            "gps_time": df_gps.loc[nearest_idx, "gps_time"]
        })

matched_infra_df = pd.DataFrame(matched_rows)

print("Matched travelled infrastructure points:", matched_infra_df.shape)
if not matched_infra_df.empty:
    print(matched_infra_df["Category"].value_counts())
    print(matched_infra_df.head())
else:
    print("No infrastructure points matched. Increase threshold slightly.")
# =========================

# =========================
# Segment vibration data
# =========================

dt_vibration = 0.002
segment_duration_seconds = 10
segment_length = int(segment_duration_seconds / dt_vibration)

segments = []
segment_info = []

num_segments = len(df_vibration) // segment_length

for i in range(num_segments):
    start_idx = i * segment_length
    end_idx = (i + 1) * segment_length

    seg = df_vibration.iloc[start_idx:end_idx][["vibration1", "vibration2"]].values
    seg_start_time = start_idx * dt_vibration
    seg_end_time = (end_idx - 1) * dt_vibration
    seg_mid_time = (seg_start_time + seg_end_time) / 2

    segments.append(seg)
    segment_info.append({
        "segment_index": i,
        "start_time": seg_start_time,
        "end_time": seg_end_time,
        "mid_time": seg_mid_time
    })

segments = np.array(segments)
segment_info_df = pd.DataFrame(segment_info)

print("Segments shape:", segments.shape)
print(segment_info_df.head())
# =========================

# =========================
# Label segments
# =========================

segment_labels = []

# time threshold in seconds
label_time_threshold = 5

for _, seg_row in segment_info_df.iterrows():
    if matched_infra_df.empty:
        segment_labels.append("Other")
        continue

    time_diffs = np.abs(matched_infra_df["gps_time"] - seg_row["mid_time"])
    nearest_idx = time_diffs.idxmin()
    nearest_time_diff = time_diffs.loc[nearest_idx]

    if nearest_time_diff <= label_time_threshold:
        label = matched_infra_df.loc[nearest_idx, "Category"]
    else:
        label = "Other"

    segment_labels.append(label)

segment_info_df["Label"] = segment_labels

print(segment_info_df["Label"].value_counts())
print(segment_info_df.head(20))
# =========================


# =========================
# Label GPS points for map visualization
# =========================

df_gps["Label"] = "Other"

if not matched_infra_df.empty:
    for _, row in matched_infra_df.iterrows():
        gps_idx = int(row["NearestGPSIndex"])
        df_gps.loc[gps_idx, "Label"] = row["Category"]

print(df_gps["Label"].value_counts())
# =========================

# =========================
# Interactive GPS map
# =========================

map_fig = go.Figure()

# Full GPS path as a thin line
map_fig.add_trace(go.Scattermapbox(
    lat=df_gps["Latitude"],
    lon=df_gps["Longitude"],
    mode="lines",
    line=dict(width=2, color="gray"),
    name="Train path"
))

# Only labeled turnout points
turnout_points = df_gps[df_gps["Label"] == "Turnout"]

map_fig.add_trace(go.Scattermapbox(
    lat=turnout_points["Latitude"],
    lon=turnout_points["Longitude"],
    mode="markers",
    marker=dict(size=9, color="red"),
    text=["Turnout"] * len(turnout_points),
    name="Turnout"
))

map_fig.update_layout(
    mapbox_style="open-street-map",
    mapbox=dict(
        center=dict(
            lat=df_gps["Latitude"].mean(),
            lon=df_gps["Longitude"].mean()
        ),
        zoom=10
    ),
    title="GPS Path with Matched Turnout Points",
    height=650
)
# =========================

# =========================
# Empty vibration figure
# =========================

vib_empty_fig = go.Figure()
vib_empty_fig.update_layout(
    title="Vibration Signal",
    xaxis_title="Time (s)",
    yaxis_title="Acceleration"
)
# =========================

# =========================
# Dash app
# =========================

app = dash.Dash(__name__)

app.layout = html.Div([
    html.Div([
        dcc.Graph(id="gps-map", figure=map_fig)
    ], style={"width": "48%", "display": "inline-block", "vertical-align": "top"}),

    html.Div([
        dcc.Graph(id="vibration-plot", figure=vib_empty_fig)
    ], style={"width": "48%", "display": "inline-block", "vertical-align": "top"})
])
# =========================

# =========================
# Callback
# =========================

@app.callback(
    Output("vibration-plot", "figure"),
    Input("gps-map", "clickData")
)
def update_vibration_plot(clickData):
    # No click yet
    if clickData is None:
        return vib_empty_fig

    try:
        # Safe read of clicked point index
        point_index = int(clickData["points"][0]["pointIndex"])

        # Safety check
        if point_index not in df_gps.index:
            fig = go.Figure()
            fig.update_layout(
                title=f"Invalid GPS point index: {point_index}",
                xaxis_title="Time (s)",
                yaxis_title="Acceleration"
            )
            return fig

        gps_time = df_gps.loc[point_index, "gps_time"]
        gps_label = df_gps.loc[point_index, "Label"]

        # No vibration segments
        if len(segments) == 0:
            fig = go.Figure()
            fig.update_layout(
                title="No vibration data available",
                xaxis_title="Time (s)",
                yaxis_title="Acceleration"
            )
            return fig

        # Find nearest segment by time
        time_diffs = np.abs(segment_info_df["mid_time"] - gps_time)
        nearest_segment_idx = int(time_diffs.idxmin())

        # Safety check
        if nearest_segment_idx >= len(segments):
            nearest_segment_idx = len(segments) - 1

        selected_segment = segments[nearest_segment_idx]

        # Build x-axis based on actual selected segment length
        time_axis = np.arange(selected_segment.shape[0]) * dt_vibration

        vib_fig = go.Figure()
        vib_fig.add_trace(go.Scatter(
            x=time_axis,
            y=selected_segment[:, 0],
            mode="lines",
            name="Vibration Channel 1"
        ))

        vib_fig.add_trace(go.Scatter(
            x=time_axis,
            y=selected_segment[:, 1],
            mode="lines",
            name="Vibration Channel 2"
        ))

        vib_fig.update_layout(
            title=f"Vibration Segment | GPS point {point_index} | Label: {gps_label}",
            xaxis_title="Time (s)",
            yaxis_title="Acceleration"
        )

        return vib_fig

    except Exception as e:
        fig = go.Figure()
        fig.update_layout(
            title=f"Callback error: {str(e)}",
            xaxis_title="Time (s)",
            yaxis_title="Acceleration"
        )
        return fig
# =========================
if __name__ == "__main__":
    app.run_server(debug=True, port=8060)

GPS shape: (36000, 6)
    Latitude  Longitude   Speed  Satellites  PointIndex  gps_time
0  60.483572  15.425976  0.3885           5           0      0.00
1  60.483571  15.425976  0.4440           5           1      0.05
2  60.483571  15.425976  0.4070           5           2      0.10
3  60.483571  15.425977  0.2590           5           3      0.15
4  60.483571  15.425977  0.1480           5           4      0.20
Vibration shape: (36000003, 3)
   vibration1  vibration2  timestamp
0    0.076299    2.060062      0.000
1    0.091558    2.060062      0.002
2    0.095373    2.071507      0.004
3    0.106818    2.044803      0.006
4    0.083928    2.071507      0.008
Infrastructure shape: (120, 3)
Category
Turnout       75
Bridge        25
Rail-joint    20
Name: count, dtype: int64
Matched travelled infrastructure points: (0, 0)
No infrastructure points matched. Increase threshold slightly.
Segments shape: (7200, 5000, 2)
   segment_index  start_time  end_time  mid_time
0              0    

C:\Users\Daso-PC\AppData\Local\Temp\ipykernel_28532\1899949100.py:226: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\Daso-PC\AppData\Local\Temp\ipykernel_28532\1899949100.py:237: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [2]:
import os
import pandas as pd
import numpy as np

def process_trip(trip_path, infra_df, distance_threshold=0.002, label_time_threshold=5):
    # File paths
    gps_lat_file = os.path.join(trip_path, "GPS.latitude.csv")
    gps_lon_file = os.path.join(trip_path, "GPS.longitude.csv")
    gps_speed_file = os.path.join(trip_path, "GPS.speed.csv")
    gps_sat_file = os.path.join(trip_path, "GPS.satellites.csv")
    vib1_file = os.path.join(trip_path, "CH1_ACCEL1Z1.csv")
    vib2_file = os.path.join(trip_path, "CH2_ACCEL1Z2.csv")

    # Skip folder if required files are missing
    required_files = [
        gps_lat_file, gps_lon_file, gps_speed_file, gps_sat_file,
        vib1_file, vib2_file
    ]
    if not all(os.path.exists(f) for f in required_files):
        print(f"Skipping {os.path.basename(trip_path)} - missing files")
        return None

    # Load GPS data
    lat_df = pd.read_csv(gps_lat_file, header=None, names=["Latitude"])
    lon_df = pd.read_csv(gps_lon_file, header=None, names=["Longitude"])
    speed_df = pd.read_csv(gps_speed_file, header=None, names=["Speed"])
    sat_df = pd.read_csv(gps_sat_file, header=None, names=["Satellites"])

    df_gps = pd.concat([lat_df, lon_df, speed_df, sat_df], axis=1)
    df_gps = df_gps.apply(pd.to_numeric, errors="coerce")
    df_gps = df_gps.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

    if df_gps.empty:
        print(f"Skipping {os.path.basename(trip_path)} - empty GPS data after cleaning")
        return None

    df_gps["PointIndex"] = df_gps.index
    df_gps["gps_time"] = df_gps.index * 0.05

    # Load vibration data
    vib1_df = pd.read_csv(vib1_file, header=None, names=["vibration1"])
    vib2_df = pd.read_csv(vib2_file, header=None, names=["vibration2"])

    df_vibration = pd.concat([vib1_df, vib2_df], axis=1)
    df_vibration = df_vibration.apply(pd.to_numeric, errors="coerce")
    df_vibration = df_vibration.dropna().reset_index(drop=True)
    df_vibration["timestamp"] = df_vibration.index * 0.002

    if df_vibration.empty:
        print(f"Skipping {os.path.basename(trip_path)} - empty vibration data after cleaning")
        return None

    # Match infrastructure points to GPS path
    def find_nearest_gps_point(lat, lon, gps_df):
        distances = np.sqrt((gps_df["Latitude"] - lat) ** 2 + (gps_df["Longitude"] - lon) ** 2)
        nearest_idx = distances.idxmin()
        nearest_distance = distances.loc[nearest_idx]
        return nearest_idx, nearest_distance

    matched_rows = []
    for _, infra_row in infra_df.iterrows():
        nearest_idx, nearest_distance = find_nearest_gps_point(
            infra_row["Latitude"],
            infra_row["Longitude"],
            df_gps
        )

        if nearest_distance <= distance_threshold:
            matched_rows.append({
                "Latitude": infra_row["Latitude"],
                "Longitude": infra_row["Longitude"],
                "Category": infra_row["Category"],
                "NearestGPSIndex": nearest_idx,
                "NearestDistance": nearest_distance,
                "gps_time": df_gps.loc[nearest_idx, "gps_time"]
            })

    matched_infra_df = pd.DataFrame(matched_rows)

    # Segment vibration data
    dt_vibration = 0.002
    segment_duration_seconds = 10
    segment_length = int(segment_duration_seconds / dt_vibration)

    segments = []
    segment_info = []

    num_segments = len(df_vibration) // segment_length

    if num_segments == 0:
        print(f"Skipping {os.path.basename(trip_path)} - not enough vibration samples for one segment")
        return None

    for i in range(num_segments):
        start_idx = i * segment_length
        end_idx = (i + 1) * segment_length

        seg = df_vibration.iloc[start_idx:end_idx][["vibration1", "vibration2"]].values
        seg_start_time = start_idx * dt_vibration
        seg_end_time = (end_idx - 1) * dt_vibration
        seg_mid_time = (seg_start_time + seg_end_time) / 2

        segments.append(seg)
        segment_info.append({
            "segment_index": i,
            "start_time": seg_start_time,
            "end_time": seg_end_time,
            "mid_time": seg_mid_time
        })

    segment_info_df = pd.DataFrame(segment_info)

    # Label segments
    labels = []
    for _, seg_row in segment_info_df.iterrows():
        if matched_infra_df.empty:
            labels.append("Other")
            continue

        time_diffs = np.abs(matched_infra_df["gps_time"] - seg_row["mid_time"])
        nearest_idx = time_diffs.idxmin()
        nearest_time_diff = time_diffs.loc[nearest_idx]

        if nearest_time_diff <= label_time_threshold:
            labels.append(matched_infra_df.loc[nearest_idx, "Category"])
        else:
            labels.append("Other")

    segment_info_df["Label"] = labels
    segment_info_df["TripFolder"] = os.path.basename(trip_path)

    return {
        "trip_name": os.path.basename(trip_path),
        "gps_df": df_gps,
        "matched_infra_df": matched_infra_df,
        "segment_info_df": segment_info_df
    }

In [3]:
data2_root = r"Data 2\Data 2"

trip_folders = [
    os.path.join(data2_root, folder)
    for folder in os.listdir(data2_root)
    if os.path.isdir(os.path.join(data2_root, folder))
]

all_results = []
all_segment_info = []
all_matched_infra = []

for trip_path in trip_folders:
    print(f"Processing: {os.path.basename(trip_path)}")
    result = process_trip(trip_path, infra_df)

    if result is not None:
        all_results.append(result)
        all_segment_info.append(result["segment_info_df"])
        all_matched_infra.append(result["matched_infra_df"])

combined_segment_info_df = pd.concat(all_segment_info, ignore_index=True) if all_segment_info else pd.DataFrame()
combined_matched_infra_df = pd.concat(all_matched_infra, ignore_index=True) if all_matched_infra else pd.DataFrame()

print("\nAll trips finished.")
print("Combined segment labels:")
print(combined_segment_info_df["Label"].value_counts())

print("\nCombined matched infrastructure:")
if not combined_matched_infra_df.empty:
    print(combined_matched_infra_df["Category"].value_counts())
else:
    print("No matched infrastructure points found.")

Processing: 2024-12-08 02-00-00 (1)
Skipping 2024-12-08 02-00-00 (1) - missing files
Processing: 2024-12-08 04-00-00 (1)
Skipping 2024-12-08 04-00-00 (1) - missing files
Processing: 2024-12-08 06-00-00 (1)
Skipping 2024-12-08 06-00-00 (1) - missing files
Processing: 2024-12-08 08-00-00 (1)
Skipping 2024-12-08 08-00-00 (1) - missing files
Processing: 2024-12-08 10-00-00 (1)
Skipping 2024-12-08 10-00-00 (1) - empty GPS data after cleaning
Processing: 2024-12-10 10-00-00 (1)
Processing: 2024-12-10 12-00-00 (1)
Processing: 2024-12-10 14-00-00 (1)
Processing: 2024-12-10 16-00-00 (1)
Processing: 2024-12-10 18-00-00 (1)
Processing: 2024-12-10 20-00-00 (1)
Processing: 2024-12-10 22-00-00 (1)
Processing: 2024-12-11 00-00-00 (1)
Processing: 2024-12-11 02-00-00 (1)
Processing: 2024-12-11 04-00-00 (1)
Processing: 2024-12-11 06-00-00 (1)
Processing: 2024-12-11 08-00-00 (1)
Processing: 2024-12-11 10-00-00 (1)
Processing: 2024-12-11 12-00-00 (1)
Processing: 2024-12-11 14-00-00 (1)
Processing: 2024-12

In [4]:
print("Segment labels count:")
print(combined_segment_info_df["Label"].value_counts())

print("\nMatched infrastructure count:")
print(combined_matched_infra_df["Category"].value_counts())

Segment labels count:
Label
Other         694822
Turnout          222
Bridge           133
Rail-joint        57
Name: count, dtype: int64

Matched infrastructure count:
Category
Turnout       523
Bridge        176
Rail-joint    140
Name: count, dtype: int64
